In [16]:
import pandas as pd
from sklearn.metrics import accuracy_score,f1_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.model_selection import train_test_split

In [17]:
class FlexibleNeuralNetwork:
    def __init__(self, architecture):
        """
        Create a neural network with custom architecture.
        
        Args:
            architecture: List of integers defining layer sizes
                         Example: [10, 20, 15, 3] = 10 inputs, 2 hidden layers, 3 outputs
        """
        self.architecture = architecture
        self.num_layers = len(architecture)
        self.weights = []
        self.biases = []
        
        # Initialize weights and biases
        for i in range(len(architecture) - 1):
            w = np.random.randn(architecture[i], architecture[i+1]) * np.sqrt(2.0 / architecture[i])
            b = np.zeros((1, architecture[i+1]))
            self.weights.append(w)
            self.biases.append(b)
    
    # Activation functions
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    def sigmoid_derivative(self, x):
        return x * (1 - x)
    
    def tanh(self, x):
        return np.tanh(x)
    
    def tanh_derivative(self, x):
        return 1 - x**2
    
    def relu(self, x):
        return np.maximum(0, x)
    
    def relu_derivative(self, x):
        return (x > 0).astype(float)
    
    def softmax(self, x):
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)
    
    def activate(self, x, activation):
        activations = {
            'sigmoid': self.sigmoid,
            'tanh': self.tanh,
            'relu': self.relu,
            'softmax': self.softmax
        }
        return activations.get(activation, self.sigmoid)(x)
    
    def get_derivative(self, x, activation):
        derivatives = {
            'sigmoid': self.sigmoid_derivative,
            'tanh': self.tanh_derivative,
            'relu': self.relu_derivative
        }
        return derivatives.get(activation, self.sigmoid_derivative)(x)
    
    def forward(self, X, activation='relu'):
        """Forward propagation through all layers"""
        self.activations = [X]
        self.z_values = []
        current_input = X
        
        for i in range(len(self.weights)):
            z = np.dot(current_input, self.weights[i]) + self.biases[i]
            self.z_values.append(z)
            
            # Use softmax for last layer if multi-class
            if i == len(self.weights) - 1 and self.architecture[-1] > 1:
                a = self.softmax(z)
            else:
                a = self.activate(z, activation)
            
            self.activations.append(a)
            current_input = a
        
        return self.activations[-1]
    
    def compute_loss(self, y_true, y_pred, loss_type='mse'):
        """Calculate loss"""
        if loss_type == 'mse':
            return np.mean((y_true - y_pred) ** 2)
        elif loss_type == 'binary_crossentropy':
            y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)
            return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
        elif loss_type == 'categorical_crossentropy':
            y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)
            return -np.mean(np.sum(y_true * np.log(y_pred), axis=1))
        return np.mean((y_true - y_pred) ** 2)
    
    def backward(self, X, y, activation='relu', loss_type='mse'):
        m = X.shape[0]
        self.d_weights = []
        self.d_biases = []
    
        if loss_type == 'categorical_crossentropy':
            delta = self.activations[-1] - y
        else:
            delta = (self.activations[-1] - y) * self.get_derivative(
                self.activations[-1], activation
            )
    
        for i in range(len(self.weights) - 1, -1, -1):
            dW = np.dot(self.activations[i].T, delta) / m
            db = np.sum(delta, axis=0, keepdims=True) / m
    
            self.d_weights.insert(0, dW)
            self.d_biases.insert(0, db)
    
            if i > 0:
                delta = np.dot(delta, self.weights[i].T) * self.get_derivative(
                    self.activations[i], activation
                )

    def update_weights(self, learning_rate):
        for i in range(len(self.weights)):
            self.weights[i] -= learning_rate * self.d_weights[i]
            self.biases[i] -= learning_rate * self.d_biases[i]

    
    def train(self, X, y, epochs=1000, learning_rate=0.01, 
              activation='relu', loss_type='mse', verbose=True):
        """Train the network"""
        losses = []
        
        for epoch in range(epochs):
            output = self.forward(X, activation)
            loss = self.compute_loss(y, output, loss_type)
            losses.append(loss)
            
            self.backward(X, y, activation, loss_type)
            self.update_weights(learning_rate)
            
            if verbose and epoch % (epochs // 10) == 0:
                print(f"Epoch {epoch}/{epochs}, Loss: {loss:.6f}")
        
        return losses
    
    def predict(self, X, activation='relu'):
        """Make predictions"""
        output = self.forward(X, activation)
        
        # Binary classification
        if output.shape[1] == 1:
            return (output > 0.5).astype(int)
        
        # Multi-class classification
        return np.argmax(output, axis=1).reshape(-1, 1)
    
    def evaluate(self, X, y, activation='relu'):
        """Evaluate model"""
        predictions = self.predict(X, activation)
        accuracy = np.mean(predictions == y)
        return accuracy


# Example 1: XOR Problem (Binary Classification)
print("="*50)
print("Example 1: XOR Problem")
print("="*50)

X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([[0], [1], [1], [0]])

nn_xor = FlexibleNeuralNetwork([2, 6, 1])
nn_xor.train(X_xor, y_xor, epochs=5000, learning_rate=0.5, 
             activation='sigmoid', loss_type='mse')

print("\nPredictions:")
for x, pred in zip(X_xor, nn_xor.predict(X_xor, 'sigmoid')):
    print(f"{x} -> {int(pred[0])}")


# Example 2: Multi-class Classification
print("\n" + "="*50)
print("Example 3: Multi-class Classification")
print("="*50)

# Generate 3-class dataset
from sklearn.datasets import make_classification

X_multi, y_multi_labels = make_classification(
    n_samples=300, n_features=4, n_informative=3,
    n_redundant=0, n_classes=3, n_clusters_per_class=1, random_state=42
)

# One-hot encode labels
y_multi = np.zeros((y_multi_labels.size, 3))
y_multi[np.arange(y_multi_labels.size), y_multi_labels] = 1

nn_multi = FlexibleNeuralNetwork([4, 10, 6, 3])
nn_multi.train(X_multi, y_multi, epochs=1000, learning_rate=0.01,
               activation='relu', loss_type='categorical_crossentropy')

accuracy = nn_multi.evaluate(X_multi, y_multi_labels.reshape(-1, 1), 'relu')
print(f"\nAccuracy: {accuracy:.2%}")

Example 1: XOR Problem
Epoch 0/5000, Loss: 0.379864
Epoch 500/5000, Loss: 0.201720
Epoch 1000/5000, Loss: 0.078461
Epoch 1500/5000, Loss: 0.028358
Epoch 2000/5000, Loss: 0.014829
Epoch 2500/5000, Loss: 0.009545
Epoch 3000/5000, Loss: 0.006884
Epoch 3500/5000, Loss: 0.005322
Epoch 4000/5000, Loss: 0.004308
Epoch 4500/5000, Loss: 0.003603

Predictions:
[0 0] -> 0
[0 1] -> 1
[1 0] -> 1
[1 1] -> 0

Example 3: Multi-class Classification
Epoch 0/1000, Loss: 1.364277
Epoch 100/1000, Loss: 0.816616
Epoch 200/1000, Loss: 0.686774
Epoch 300/1000, Loss: 0.601374
Epoch 400/1000, Loss: 0.541831
Epoch 500/1000, Loss: 0.502116
Epoch 600/1000, Loss: 0.471922
Epoch 700/1000, Loss: 0.448413
Epoch 800/1000, Loss: 0.430012
Epoch 900/1000, Loss: 0.414998

Accuracy: 84.33%

Example 3: Load Your Own CSV

To use your own dataset:

1. Prepare CSV with features in columns, target in last column
2. Load and train:

    data = pd.read_csv('your_data.csv')
    X = data.iloc[:, :-1].values
    y = data.iloc[:, -1].

In [18]:
df = pd.read_csv('../Boosting/dataset/heart_cleveland.csv')

In [19]:
X = df.drop('condition', axis =1)

In [20]:
y = df['condition']

In [21]:
X

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,69,1,0,160,234,1,2,131,0,0.1,1,1,0
1,69,0,0,140,239,0,0,151,0,1.8,0,2,0
2,66,0,0,150,226,0,0,114,0,2.6,2,0,0
3,65,1,0,138,282,1,2,174,0,1.4,1,1,0
4,64,1,0,110,211,0,2,144,1,1.8,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
292,40,1,3,152,223,0,0,181,0,0.0,0,0,2
293,39,1,3,118,219,0,0,140,0,1.2,1,0,2
294,35,1,3,120,198,0,0,130,1,1.6,1,0,2
295,35,0,3,138,183,0,0,182,0,1.4,0,0,0


In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [24]:
X = X_train.values
y = y_train.values.reshape(-1, 1)
#I turn this into NumPy array as my fnn works with them

In [39]:
X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)
nn = FlexibleNeuralNetwork([13,16,8,1])

In [44]:
nn.train(
    X,
    y,
    epochs=5000,
    learning_rate=0.1,
    activation="sigmoid",
    loss_type="binary_crossentropy"
)

Epoch 0/5000, Loss: 1.723098
Epoch 500/5000, Loss: 0.598363
Epoch 1000/5000, Loss: 0.470109
Epoch 1500/5000, Loss: 0.398871
Epoch 2000/5000, Loss: 0.364591
Epoch 2500/5000, Loss: 0.344408
Epoch 3000/5000, Loss: 0.330864
Epoch 3500/5000, Loss: 0.320963
Epoch 4000/5000, Loss: 0.313307
Epoch 4500/5000, Loss: 0.307193


[np.float64(1.7230977898065587),
 np.float64(1.720771511190414),
 np.float64(1.7184365403613584),
 np.float64(1.7160928205094657),
 np.float64(1.7137402943521531),
 np.float64(1.7113789041304586),
 np.float64(1.7090085916053184),
 np.float64(1.7066292980538613),
 np.float64(1.7042409642656995),
 np.float64(1.7018435305392392),
 np.float64(1.6994369366779962),
 np.float64(1.69702112198693),
 np.float64(1.6945960252687975),
 np.float64(1.6921615848205185),
 np.float64(1.689717738429571),
 np.float64(1.6872644233704084),
 np.float64(1.684801576400902),
 np.float64(1.6823291337588204),
 np.float64(1.6798470311583351),
 np.float64(1.6773552037865698),
 np.float64(1.6748535863001872),
 np.float64(1.6723421128220188),
 np.float64(1.6698207169377466),
 np.float64(1.6672893316926325),
 np.float64(1.6647478895883074),
 np.float64(1.6621963225796144),
 np.float64(1.6596345620715263),
 np.float64(1.6570625389161227),
 np.float64(1.6544801834096496),
 np.float64(1.6518874252896503),
 np.float64(1.6

In [75]:

X_train_np = X_train.values
y_train_np = y_train.values.reshape(-1, 1)

X_test_np = X_test.values
y_test_np = y_test.values.reshape(-1, 1)

train_mean = X_train_np.mean(axis=0)
train_std = X_train_np.std(axis=0) + 1e-8

X_train_np = (X_train_np - train_mean) / train_std
X_test_np = (X_test_np - train_mean) / train_std

nn = FlexibleNeuralNetwork([X_train_np.shape[1], 16, 8, 1])
nn.train(X_train_np, y_train_np, epochs=2000, learning_rate=0.01,
         activation="relu", loss_type="binary_crossentropy")

# Evaluate
accuracy = nn.evaluate(X_test_np, y_test_np, activation="relu")
print("Test accuracy:", accuracy)

Epoch 0/2000, Loss: 6.586567
Epoch 200/2000, Loss: 6.274557
Epoch 400/2000, Loss: 5.270023
Epoch 600/2000, Loss: 4.165115
Epoch 800/2000, Loss: 2.734472
Epoch 1000/2000, Loss: 1.736570
Epoch 1200/2000, Loss: 1.624388
Epoch 1400/2000, Loss: 1.435824
Epoch 1600/2000, Loss: 1.398997
Epoch 1800/2000, Loss: 1.333222
Test accuracy: 0.7333333333333333
